# TF-IDF Search Engine
We build a TF-IDF based search engine for customer support tweets.

The goal is to retrieve the most relevant tweets given a user query.

In [8]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [9]:
df = pd.read_csv("../Data/final_tweets.csv")

df.shape

(59184, 4)

In [10]:
df.columns

Index(['tweet_id', 'text', 'clean_text', 'company'], dtype='str')

In [11]:
search_df = df[["tweet_id", "clean_text", "company"]].copy()

search_df.head()

,tweet_id,clean_text,company
0,277379,another great set of flights thank to tmobilew...,delta
1,2488664,why is my iphone automatically going on mute,applesupport
2,2383194,since yesterday i havent received any update a...,uber_support
3,1610886,there is no email from days just asked to wait...,amazonhelp
4,2763258,thanks for responding will call as soon as i g...,southwestair


In [12]:
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(search_df["clean_text"])

tfidf_matrix.shape

(59184, 10000)

In [13]:
def search_tweets(query, top_k=5):
    # convert query into vector
    query_vec = tfidf.transform([query])
    
    # compute similarity
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # get top results
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = search_df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    
    return results

In [14]:
queries = [
    "flight delayed customer service",
    "refund for cancelled order",
    "app not working",
    "bad customer support",
    "lost baggage complaint"
]

for q in queries:
    print("=" * 80)
    print("Query:", q)
    display(search_tweets(q, top_k=5))

Query: flight delayed customer service


,tweet_id,clean_text,company,score
2472,1799160,flight was already delayed on top of it poor c...,americanair,0.812467
30389,565503,why is flight delayed,americanair,0.725899
42254,975859,another delayed flight nomoredeltaatjfkiamdone...,delta,0.725899
51365,1370386,when your flight is delayed,southwestair,0.725899
22745,2192623,my flight is delayed thanks,southwestair,0.627027


Query: refund for cancelled order


,tweet_id,clean_text,company,score
29980,242928,cancelled my order,amazonhelp,0.815143
44897,1736159,order cancelled,amazonhelp,0.815143
4601,2192481,you cancelled the order not me,amazonhelp,0.815143
7423,1332117,cancelled,amazonhelp,0.631022
18825,1137720,its cancelled,amazonhelp,0.631022


Query: app not working


,tweet_id,clean_text,company,score
53628,2884478,yet again app not working,uber_support,1.000000
1376,2839586,still not working,southwestair,0.755069
14399,2106689,still not working,virgintrains,0.755069
25021,26886,its not working,amazonhelp,0.755069
13324,437019,still not working,applesupport,0.755069


Query: bad customer support


,tweet_id,clean_text,company,score
11725,1707849,too bad,southwestair,0.646984
19359,2552007,just lost due to bad customer support represen...,amazonhelp,0.638996
1329,1098833,worst customer support ever,delta,0.575381
287,382575,customer for years worst customer support ever,amazonhelp,0.546589
21115,21909,your customer support told me it has battery,amazonhelp,0.518513


Query: lost baggage complaint


,tweet_id,clean_text,company,score
57075,1906961,could you advise me on my lost baggage claim,delta,0.547610
7533,1117228,always lovely when baggage is lost or put on t...,americanair,0.491680
2331,1111612,why cant you locate lost baggage are you kiddi...,southwestair,0.487360
3722,983788,how much time i complaint,amazonhelp,0.487320
38200,2799767,next issue will be my baggage find it please n...,americanair,0.471627


TF-IDF search worked well when the same query words were found in the tweets. For example, searches like “app not working”, “refund for cancelled order”, and “flight delayed customer service” returned closely related tweets.

However, TF-IDF mostly depends on exact word matches. This means it may miss useful tweets if people use different words for the same problem.